# 00 publications (g1)

Every list we have seen, one row per distinct file. What you should see: 8 rows, one per source; `as_of_ok` and `hash_ok` true on every row; EirGrid as of 2026-08-05, ESB as of 2026-07-01. `published_at` is the server's Last-Modified (an upload time, so labelled estimated).

In [ ]:
import subprocess

from iegrid import store
from iegrid.checks import check_publications, publication_lag, table

snap = store.latest_snapshot()
sha = subprocess.run(["git", "-C", str(store.ROOT), "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True, check=True).stdout.strip()
con = store.connect_read_only(snap)
print(f"snapshot {snap.name}  git {sha}")

In [ ]:
print(table(con, """
    select source_id, as_of, published_at, published_at_basis, vintage_quality,
           round(size_bytes / 1024) as kb, left(sha256, 12) as sha
    from publication order by source_id, as_of
"""))

## done-when: as-of in the PDF matches the date in the file name, raw file hash matches

In [ ]:
checks = check_publications(con)  # live editions
for c in checks:
    print(f"{c.source_id:34} pdf {c.as_of}  name {c.as_of_from_name}  as_of_ok {c.as_of_ok}  hash_ok {c.hash_ok}")
assert checks, "no publications stored"
assert all(c.as_of_ok for c in checks), "as-of in the PDF disagrees with the file name"
assert all(c.hash_ok for c in checks), "a raw file is missing or changed"
print(f"\n{len(checks)} publications, all checks pass")

# archived editions (g3): hashes must match; date differences are shown, not asserted
old = check_publications(con, archived=True)
assert all(c.hash_ok for c in old), "an archived raw file is missing or changed"
differ = [c for c in old if c.as_of_from_name and not c.as_of_ok]
print(f"{len(old)} archived editions, hashes ok; {len(differ)} where the text date differs "
      f"from a date read off the file name (the text is the authority):")
for c in differ:
    print(f"  {c.source_id:32} text {c.as_of}  name {c.as_of_from_name}")

## lag: how long after the as-of date each list went up

In [ ]:
print(publication_lag(con))

## exhibit: how fresh the lists are

In [ ]:
from iegrid import exhibits

fig = exhibits.publication_lag(con)
exhibits.save(fig, "g1_publication_lag")